# Bank Customer Churn Analysis
**HCL Training Project | ABES B.Tech Data Analyst | Project No. 8**

This notebook loads the supplied CSV dataset, performs data-quality checks, cleans records, calculates KPIs and explores churn by age, geography and product count.

**Important:** The supplied dataset is synthetic training data, not real bank data.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
df = pd.read_csv(DATA_DIR / "bank_churn_raw.csv")
df.head()

## 2. Inspect the dataset

In [ ]:
print("Shape:", df.shape)
print("\\nData types:\\n", df.dtypes)
print("\\nMissing values:\\n", df.isna().sum())
print("Duplicate customer IDs:", df["CustomerId"].duplicated().sum())
display(df.describe(include="all").T)

## 3. Data cleaning

In [ ]:
clean = df.drop_duplicates(subset="CustomerId").copy()

# Age outside 18-100 is treated as invalid for this analysis.
invalid_age_rows = clean.loc[~clean["Age"].between(18, 100)].copy()
clean = clean[clean["Age"].between(18, 100)].copy()

# Fill missing geography with the most common geography.
clean["Geography"] = clean["Geography"].fillna(clean["Geography"].mode()[0])

# Zero balance is retained; it is not automatically a data error.
print("Invalid-age rows excluded:", len(invalid_age_rows))
print("Cleaned shape:", clean.shape)
print("Remaining missing values:", clean.isna().sum().sum())
clean.to_csv(DATA_DIR / "bank_churn_clean.csv", index=False)

## 4. KPI calculations

In [ ]:
total_customers = clean["CustomerId"].nunique()
churned_customers = int(clean["Exited"].sum())
retained_customers = total_customers - churned_customers
churn_rate = churned_customers / total_customers * 100
retention_rate = 100 - churn_rate

kpis = pd.DataFrame({
    "KPI": ["Total Customers", "Churned Customers", "Retained Customers",
            "Churn Rate (%)", "Retention Rate (%)"],
    "Value": [total_customers, churned_customers, retained_customers,
              round(churn_rate, 2), round(retention_rate, 2)]
})
display(kpis)

## 5. Churn by geography

In [ ]:
geo = clean.groupby("Geography").agg(
    Customers=("CustomerId", "count"), Churned=("Exited", "sum"),
    ChurnRate=("Exited", "mean")
)
geo["ChurnRate"] = (geo["ChurnRate"] * 100).round(2)
display(geo)

sns.barplot(data=clean, x="Geography", y="Exited", errorbar=None)
plt.title("Churn rate by geography")
plt.ylabel("Churn rate")
plt.show()

## 6. Churn by number of products

In [ ]:
products = clean.groupby("NumOfProducts").agg(
    Customers=("CustomerId", "count"), Churned=("Exited", "sum"),
    ChurnRate=("Exited", "mean")
)
products["ChurnRate"] = (products["ChurnRate"] * 100).round(2)
display(products)

sns.barplot(data=clean, x="NumOfProducts", y="Exited", errorbar=None)
plt.title("Churn rate by number of products")
plt.ylabel("Churn rate")
plt.show()

## 7. Churn by age group

In [ ]:
clean["AgeGroup"] = pd.cut(
    clean["Age"], bins=[17, 30, 40, 50, 60, 100],
    labels=["18-30", "31-40", "41-50", "51-60", "61+"]
)
age = clean.groupby("AgeGroup", observed=False).agg(
    Customers=("CustomerId", "count"), Churned=("Exited", "sum"),
    ChurnRate=("Exited", "mean")
)
age["ChurnRate"] = (age["ChurnRate"] * 100).round(2)
display(age)

sns.barplot(data=clean, x="AgeGroup", y="Exited", errorbar=None)
plt.title("Churn rate by age group")
plt.ylabel("Churn rate")
plt.show()

## 8. Additional analysis: activity and zero balance

In [ ]:
activity = clean.groupby("IsActiveMember")["Exited"].mean().mul(100).round(2)
zero_balance = clean.assign(ZeroBalance=clean["Balance"].eq(0)).groupby("ZeroBalance")["Exited"].mean().mul(100).round(2)
print("Churn rate by active-member flag (%):\\n", activity)
print("\\nChurn rate by zero-balance flag (%):\\n", zero_balance)

## 9. Manual validation for three segments
The following validation prints the numerator, denominator and calculated churn rate for each geography. Use these values to cross-check the SQL query and Power BI results.

In [ ]:
validation = clean.groupby("Geography")["Exited"].agg(["sum", "count"])
validation["manual_churn_rate_pct"] = (validation["sum"] / validation["count"] * 100).round(2)
display(validation)

## 10. Conclusion and recommendations
Use the segment tables to identify the highest-churn groups in this sample. Recommendations should be framed as hypotheses to test, not as proven causes. For real deployment, repeat the analysis on a verified bank churn dataset and validate results with business stakeholders.

Suggested actions:
- Investigate service issues and feedback in high-churn segments.
- Evaluate engagement campaigns for inactive customers.
- Monitor segment churn over time.
- Compare Python, Oracle SQL and Power BI totals before final submission.